# 📗 Sesión 2 · Ejercicio A — Estudiante
## Gráfica de líneas: evolución de rotación por años de antigüedad

**Modalidad:** Guiado — completar el hueco de construcción de la gráfica  
**Tiempo:** 40 minutos  
**Dataset:** IBM HR Analytics (`hr_ibm.csv`)

---

### Objetivo

Agregar a la app un **line chart** (`px.line`) que muestre cómo varía la tasa de
rotación según los años de antigüedad del empleado en la empresa.

La gráfica usa el mismo `@reactive.calc` que ya filtra los datos — no necesitas
duplicar ningún código de filtrado.

### El hueco

El groupby ya está hecho. Tu tarea es construir el `fig = px.line(...)` con:
- Eje X: `YearsAtCompany`
- Eje Y: `Rotacion_pct`
- Título descriptivo
- Etiquetas en español

> El patrón para devolver la gráfica es el mismo que has visto: `ui.HTML(fig.to_html(...))`


## Celda 1 — Instalación y carga

In [2]:
# Instalación y carga del dataset
import pandas as pd
import plotly.express as px

from shiny import App, ui, render, reactive

df = pd.read_csv("../hr_ibm.csv")

# necesitamos una grafica del datos de la rotacion para mostrarlo

rot_antig = df.groupby("YearsAtCompany").apply(lambda x: (x["Attrition"]=="Yes").mean()*100).reset_index(name="Rotacion_pct").round(1)


print("Vista previa de rotacion por años de antigüedad")
print(rot_antig.head(15).to_string(index=False))
print()
print(f"Pico maximo de rotación: {rot_antig['Rotacion_pct'].max():.1f}%")
print(f" en el año: {rot_antig.loc[rot_antig['Rotacion_pct'].idxmax(), 'YearsAtCompany']}")

opts_dept = ["Todos"] + sorted(df["Department"].unique().tolist())
opts_attr = ["Todos", "Yes", "No"]





Vista previa de rotacion por años de antigüedad
 YearsAtCompany  Rotacion_pct
              0          36.4
              1          34.5
              2          21.3
              3          15.6
              4          17.3
              5          10.7
              6          11.8
              7          12.2
              8          11.2
              9           9.8
             10          15.0
             11           6.2
             12           0.0
             13           8.3
             14          11.1

Pico maximo de rotación: 100.0%
 en el año: 40


C:\Users\demon\AppData\Local\Temp\ipykernel_3332\3158988095.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rot_antig = df.groupby("YearsAtCompany").apply(lambda x: (x["Attrition"]=="Yes").mean()*100).reset_index(name="Rotacion_pct").round(1)


## Estructura de la app

La app tiene **tres outputs**:

| Output | Tipo | Estado |
|--------|------|--------|
| `conteo` | `render.text` | Ya dado |
| `grafica_dept` | `render.ui` | Ya dado — úsalo como modelo |
| `grafica_antig` | `render.ui` | **Tu tarea — completar el hueco** |

El `@reactive.calc` central ya está implementado — todos los outputs lo usan.
Solo tienes que construir `px.line(...)` dentro de `grafica_antig`.


## Celda 3 — App con hueco para completar

In [ ]:
import pandas as pd
import plotly.express as px

from shiny import App, ui, render, reactive

df = pd.read_csv("../hr_ibm.csv")

opts_dept = ["Todos"] + sorted(df["Department"].unique().tolist())
opts_attr = ["Todos", "Yes", "No"]

app_ui = ui.page_fluid(
    ui.h2("Evolución de rotación dentro de la Empresa"),
    ui.layout_sidebar(
        ui.sidebar(
            ui.h4("Filtros"),
            ui.input_select("dept", "Departamento", choices=opts_dept),
            ui.input_select("attrition", "Rotación", choices=opts_attr),
            ui.input_slider("edad", "Rango de edad", min=18, max=60, value=[18,60]),
        ),
        ui.output_text("conteo"),
        ui.h4("Rotación por departamento"),
        ui.output_ui("grafica_dept"),
        ui.h4("Evolución de la Rotación por años de antigüedad"),
        ui.output_ui("grafica_antig"),
    )
)

def server(input, output, session):

    @reactive.calc
    def datos_filtrados():
        d = df.copy()
        if input.dept() != "Todos" : d = d[d["Department"] == input.dept()]
        if input.attrition() != "Todos" : d = d[d["Attrition"] == input.attrition()]
        emin, emax = input.edad()
        return d[ (d["Age"] >= emin) & (d["Age"] <= emax) ]
    
    @render.text
    def conteo():
        d = datos_filtrados()
        pct = (d["Attrition"] == "Yes").mean()*100
        return f"Empleados: {len(d)}  ---  Rotacion del grupo: {pct:.1f}%"
    
    #grafica de barras para los dept
    @render.ui
    def grafica_dept():
        d = datos_filtrados()
        rot = df.groupby("Department").apply(lambda x: (x["Attrition"]=="Yes").mean()*100).reset_index(name="Rotacion_pct").round(1)
        fig = px.bar(rot, x="Department", y="Rotacion_pct", color="Department", text_auto=".1f", title="Tasa de rotación por departamento (%)")
        fig.update_layout(showlegend=False, height=320)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))
    
    @render.ui
    def grafica_antig():
        d = datos_filtrados()
        rot_antig = df.groupby("YearsAtCompany").apply(lambda x: (x["Attrition"]=="Yes").mean()*100).reset_index(name="Rotacion_pct").round(1)

        #la construccion px.line
        fig = px.line(
            rot_antig,
            x="YearsAtCompany",
            y="Rotacion_pct",
            markers=True, #puntos en cada año
            title="Tasa de rotación segun años de antigüedad (%)",
            labels={
                "YearsAtCompany" : "Años de Antigüedad",
                "Rotacion_pct" : "Rotación (%)"
            }
        )
        fig.update_layout(height=350)
        return ui.HTML(fig.to_html(include_plotlyjs="cdn"))

app = App(app_ui, server)

#solo para verlo de forma local
import threading

def _iniciar():
    app.run(host="127.0.0.1", port=8000)

hilo = threading.Thread(target=_iniciar, daemon=True)
hilo.start()
print("App corriendo en local")

        



App corriendo en local


INFO:     Started server process [34488]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:8655 - "GET / HTTP/1.1" 200 OK


INFO:     127.0.0.1:41663 - "WebSocket /websocket/" [accepted]
INFO:     connection open
C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rot = df.groupby("Department").apply(lambda x: (x["Attrition"]=="Yes").mean()*100).reset_index(name="Rotacion_pct").round(1)
C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:56: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after grou

INFO:     127.0.0.1:38277 - "GET / HTTP/1.1" 200 OK


INFO:     connection closed
INFO:     127.0.0.1:36182 - "WebSocket /websocket/" [accepted]
INFO:     connection open
C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:48: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:56: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:48: 

INFO:     127.0.0.1:38514 - "GET / HTTP/1.1" 200 OK


INFO:     127.0.0.1:6812 - "WebSocket /websocket/" [accepted]
INFO:     connection open
C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:48: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:56: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\demon\AppData\Local\Temp\ipykernel_34488\417017919.py:48: DeprecationWarning:

DataFram

## Celda 4 — Preguntas de análisis

In [ ]:
# Preguntas de análisis — usa la gráfica de líneas para responder

print("Preguntas de análisis — Ejercicio 2A")
print("=" * 45)
print()
print("P1: Observa la gráfica de líneas sin ningún filtro aplicado.")
print("    ¿En qué año de antigüedad (eje X) está el punto más alto?")
print("    ¿Cuál es el porcentaje de rotación en ese pico?")
print()
print("P2: La línea, ¿baja de forma constante o tiene altibajos?")
print("    ¿Qué interpretación de negocio le darías a ese patrón?")
print()
print("P3: Filtra por Departamento = Human Resources.")
print("    Luego cambia a Sales.")
print("    ¿En cuál la curva baja más rápido con los años?")
print()
print("Escribe tus respuestas aquí (convierte a Markdown):")
print("P1: ...")
print("P2: ...")
print("P3: ...")
